# Solar filament segmentation: PQ-evaluated U-Net (Charan)

**Settings:** Accelerator **GPU T4 x2**, **Internet on**, add the competition data (`filament-segmentation-2026`).
To train overnight use **Save Version -> Save & Run All (Commit)**; outputs (weights, metrics, `submission.csv`) land in the Output tab.

Order: setup -> prepare (~5 min) -> train (~1.5-2.5 h) -> tune post-processing on val PQ -> predict test -> save artifacts.

In [ ]:
# 1. Setup: clone the team repo (this code lives in Charan/unet on branch charan/newdatasplit)
import os, subprocess, sys
from pathlib import Path

BRANCH = "charan/newdatasplit"
REPO = "https://github.com/hsfcoulibaly/aml_project.git"
SRC = Path("/kaggle/working/aml_project")
if not SRC.exists():
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO, str(SRC)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "segmentation-models-pytorch", "pycocotools"], check=True)

CODE = SRC / "Charan" / "unet"
os.chdir(CODE)
sys.path.insert(0, str(CODE))
os.environ["UNET_WORK_DIR"] = "/kaggle/working/unet"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
WORK = Path(os.environ["UNET_WORK_DIR"])
print(subprocess.run(["git", "-C", str(SRC), "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Prepare (7-day block split, see scripts/make_split.py): preprocessed 1024 px images, per-record masks, PQ references (also prints the human-agreement ceiling)
!python scripts/prepare.py --size 1024 --workers 4 --split splits/week_split.json

In [ ]:
# Eyeball one preprocessed image and its mask
import cv2, matplotlib.pyplot as plt, pandas as pd
m = pd.read_csv(WORK / "manifest.csv", keep_default_na=False)
row = m[m.split == "train"].iloc[0]
img = cv2.imread(str(WORK / "images" / f"{row.observation_id}.png"))
mask = cv2.imread(str(WORK / "masks" / f"{row.record_id}.png"), 0)
fig, ax = plt.subplots(1, 4, figsize=(20, 5))
for i, t in enumerate(["raw", "limb-flattened", "CLAHE"]):
    ax[i].imshow(img[:, :, i], cmap="gray"); ax[i].set_title(t)
ax[3].imshow(img[:, :, 2], cmap="gray"); ax[3].imshow(mask, alpha=0.4, cmap="autumn"); ax[3].set_title(row.record_id)
for a in ax: a.axis("off")
plt.show()

In [ ]:
# 3. Train (best checkpoint by val PQ; rerun with --resume if the session is interrupted)
RUN = "unet_r34_week"
!python scripts/train.py --name {RUN} --encoder resnet34 --epochs 40 --batch 8 --workers 4 --lr 3e-4

In [ ]:
# Training curves
h = pd.read_csv(WORK / "runs" / RUN / "history.csv")
fig, ax = plt.subplots(1, 3, figsize=(18, 4))
ax[0].plot(h.epoch, h.train_loss); ax[0].set_title("train loss")
ax[1].plot(h.epoch, h.val_dice, label="Dice@512"); ax[1].plot(h.epoch, h.val_pq, label="PQ"); ax[1].legend(); ax[1].set_title("validation")
ax[2].plot(h.epoch, h.val_split_rate, label="split rate"); ax[2].plot(h.epoch, h.val_merge_rate, label="merge rate"); ax[2].legend()
plt.show()
h.tail()

In [ ]:
# 4. Tune threshold x min-area on validation PQ (with flip TTA) -> pq_config.json
!python scripts/infer.py sweep --weights {WORK}/runs/{RUN}/best.pt --tta

In [ ]:
# 5. Predict the test set -> /kaggle/working/submission.csv
!python scripts/infer.py test --weights {WORK}/runs/{RUN}/best.pt --config {WORK}/runs/{RUN}/pq_config.json --out /kaggle/working/submission.csv
sub = pd.read_csv("/kaggle/working/submission.csv")
print(sub.shape, sub.filament_id.str.rsplit("_", n=1).str[0].nunique(), "images")
sub.head()

In [ ]:
# 6. Keep the small artifacts in the Output tab (best.pt is fp16, ~49 MB); drop the large prepared data
import shutil, json
OUT = Path("/kaggle/working/unet_results"); OUT.mkdir(exist_ok=True)
for f in ["best.pt", "history.csv", "pq_config.json", "pq_sweep.csv", "val_per_image.csv", "val_dice_per_record.csv", "args.json"]:
    p = WORK / "runs" / RUN / f
    if p.exists(): shutil.copy(p, OUT / f)
print(json.dumps(json.load(open(OUT / "pq_config.json")), indent=2))
shutil.rmtree(WORK / "images", ignore_errors=True); shutil.rmtree(WORK / "masks", ignore_errors=True)
(WORK / "runs" / RUN / "last_state.pt").unlink(missing_ok=True)